### Imports

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import pandas as pd

import random
from datetime import datetime

### Config

In [2]:
colors = {
    "DQN": 'darkorange',
    "REINFORCE": 'forestgreen',
    "ODT": 'blueviolet',
    "CMA": 'turquoise',
    "CMA-ES": 'turquoise'
}

season_colors = {
    "winter": 'blue',
    "spring": 'forestgreen',
    "summer": 'gold',
    "autumn": 'darkorange'
}

exp_lists_preformatted = [
    '4000_4179.csv', '5000_5179.csv', '6000_6179.csv', '4108_4179.csv', '5108_5179.csv', '6108_6179.csv'
] # Existing formatted datasets

## Table 1

### Loading data

In [ ]:
# === v2: build Table 2 (combined_metrics) inputs from REVISED 4xxx data ===
# Replaces formatted_experiment_data/part_4. Loads the small cache written by
# build_v2_cache.py: per-car-episode in-simulation behaviour over the last 100
# episodes of the 4xxx runs. Column names match the old part_4 agent/station
# CSVs, so the table cell below is unchanged.
# Run first (on Huron):
#   python build_v2_cache.py --metrics-root /storage_1/metrics_postfix
import sys, os
sys.path.append(os.path.abspath('.'))
import importlib
import v2_data
importlib.reload(v2_data)

cumulative_agent_df, cumulative_station_df = v2_data.load_env_metrics()
print("agent rows:", len(cumulative_agent_df),
      "| station rows:", len(cumulative_station_df))
print("seasons:", sorted(cumulative_agent_df['season'].unique()))
print("algorithms:", sorted(cumulative_agent_df['algorithm'].unique()))

### Printing data for table

In [ ]:
# === v2: Table 2 (tab:combined_metrics) from the REVISED 4xxx data ===
# Distance, Peak Traffic, and Reward are regenerated from the new 4xxx runs.
# Energy [kWh] (a "kWh metric") stays on the OLD published values per the
# revision plan -- and the raw episode-level battery columns are degenerate
# (starting_battery == ending_battery), so simulation energy is not recoverable
# from them anyway. If your new runs log a usable energy signal, swap it in here.
import numpy as np

# Old published Table 2 energy [kWh] by (algorithm, season) -- kept unchanged.
OLD_ENERGY_KWH = {
    ("CMA-ES", "winter"): 15.96, ("CMA-ES", "autumn"): 21.54,
    ("CMA-ES", "spring"): 17.95, ("CMA-ES", "summer"): 22.91,
    ("DQN", "winter"): 15.83, ("DQN", "autumn"): 21.33,
    ("DQN", "spring"): 17.57, ("DQN", "summer"): 22.79,
    ("ODT", "winter"): 15.29, ("ODT", "autumn"): 20.89,
    ("ODT", "spring"): 15.67, ("ODT", "summer"): 20.51,
    ("REINFORCE", "winter"): 16.44, ("REINFORCE", "autumn"): 21.20,
    ("REINFORCE", "spring"): 17.57, ("REINFORCE", "summer"): 22.54,
}
DISPLAY = {"CMA": "CMA-ES", "DQN": "DQN", "ODT": "ODT", "REINFORCE": "REINFORCE"}

algos = sorted(cumulative_agent_df['algorithm'].unique())
seasons = ['winter', 'spring', 'autumn', 'summer']

results = []
for algo in algos:
    disp = DISPLAY.get(algo, algo)
    print(f"===== {disp} =====")
    for season in seasons:
        a = cumulative_agent_df[(cumulative_agent_df['algorithm'] == algo) &
                                (cumulative_agent_df['season'] == season)]
        s = cumulative_station_df[(cumulative_station_df['algorithm'] == algo) &
                                  (cumulative_station_df['season'] == season)]
        if a.empty:
            print(f"  {season:7} -- no data")
            continue
        dist_mean, dist_std = a['distance_traveled'].mean(), a['distance_traveled'].std()
        rew_mean, rew_std = a['reward'].mean(), a['reward'].std()
        peak = s['traffic'].max() if len(s) else float('nan')
        energy_old = OLD_ENERGY_KWH.get((disp, season), float('nan'))
        print(f"  {season:7} dist={dist_mean:6.2f}+/-{dist_std:4.2f}  "
              f"peak_traffic={peak:.0f}  reward={rew_mean:7.2f}+/-{rew_std:4.2f}  "
              f"energy[kWh](old)={energy_old}")
        results.append([disp, season, dist_mean, dist_std, peak,
                        rew_mean, rew_std, energy_old])

results_df = pd.DataFrame(results, columns=[
    "Algorithm", "Season", "Avg Distance", "Distance Std Dev", "Peak Traffic",
    "Avg Reward", "Reward Std Dev", "Energy kWh (OLD)"])
results_df.to_csv('./table_data/table_1.csv', index=False)
print("\nSaved ./table_data/table_1.csv")
print("  (Distance / Peak Traffic / Reward = NEW 4xxx;  Energy [kWh] = OLD published)")